In [ ]:
from pathlib import Path
from torchaudio.datasets import SPEECHCOMMANDS
from torch.utils.data import DataLoader
import torch
from torch.nn.functional import pad

TARGET_LENGTH = 16000  # 1 second at 16kHz

class PadOrTrim():
    def __init__(self, max_len: int = TARGET_LENGTH, pad_value: float = 0.0,):
        self.max_len = max_len
        self.pad_value = pad_value
    
    def __call__(self, w: torch.Tensor) -> torch.Tensor:
        return pad(w[:, :self.max_len], (0, max(0, self.max_len - w.shape[1])), value=self.pad_value)

class SpeechDataset(SPEECHCOMMANDS):
    """
    Drop-in replacement for torchaudio.datasets.SPEECHCOMMANDS that:
      - accepts all original constructor args (root, subset, download, etc.)
      - adds label_to_idx / idx_to_label attributes
      - adds optional transform argument to apply to waveform on-the-fly in __getitem__

    """
    def __init__(self, *args, transform=PadOrTrim(), **kwargs):
        super().__init__(*args, **kwargs)  # forwards everything to SPEECHCOMMANDS
        self.transform = transform

        # Build label maps WITHOUT loading audio: use folder names of files in _walker
        labels = sorted({Path(p).parent.name for p in self._walker})
        self.label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
        self.idx_to_label = {i: lbl for lbl, i in self.label_to_idx.items()}

    # Override __getitem__ to only return (waveform, label_idx) and apply transform.
    # If more information is needed use get_metadata from parent class.
    def __getitem__(self, n):
        
        waveform, sample_rate, label, speaker_id, utterance_number = super().__getitem__(n)
        if self.transform is not None:
            waveform = self.transform(waveform)  # expect shape [C, T]
        meta_data = {
            "sample_rate": sample_rate,
            "label": label,
            "speaker_id": speaker_id,
            "utterance_number": utterance_number
        }
        return waveform, self.label_to_idx[label], meta_data

        
dataset = SpeechDataset(root="./data", subset="training", download=True)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for wave_batch, labels, metadata in loader:
    print(wave_batch.shape)  # torch.Size([32, 1, 16000]) # (B, C, T), only one channel for this dataset
    print([dataset.idx_to_label[idx.item()] for idx in labels])  # ['yes', 'no', 'up', ...]
    break

torch.Size([32, 1, 16000])
['no', 'nine', 'happy', 'down', 'right', 'left', 'down', 'up', 'down', 'down', 'seven', 'happy', 'dog', 'cat', 'go', 'left', 'right', 'four', 'stop', 'off', 'one', 'bed', 'nine', 'stop', 'left', 'wow', 'off', 'dog', 'yes', 'zero', 'down', 'cat']


In [ ]:
class SPEECH_DATASET(SPEECHCOMMANDS):
    def __init__(self, root, subset):
        super().__init__(root, download=True)
        self.label_to_idx = {label: i for i, label in enumerate(sorted(set([label for _, _, label, _, _ in self])))}
        self.idx_to_label = {i: label for label, i in self.label_to_idx.items()}

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    waveforms, labels, speakers, utts = zip(*batch)
    # Right-pad all to same length as longest
    waveforms = pad_sequence(waveforms, batch_first=True)  # [B, T, C] or [B, 1, T]
    # If shape is [B, T, 1], move channel dimension
    if waveforms.ndim == 3 and waveforms.shape[-1] == 1:
        waveforms = waveforms.permute(0, 2, 1)
    return waveforms, labels, speakers, utts

In [26]:
import torch.nn.functional as F

class PadOrTrimTo:
    def __init__(self, num_samples: int, pad_right: bool = True):
        self.num_samples = num_samples
        self.pad_right = pad_right

    def __call__(self, waveform: torch.Tensor) -> torch.Tensor:
        # waveform: [C, T]
        C, T = waveform.shape
        if T == self.num_samples:
            return waveform
        if T > self.num_samples:
            # Trim
            return waveform[:, :self.num_samples]
        # Pad
        pad_total = self.num_samples - T
        if self.pad_right:
            pad = (0, pad_total)  # (pad_left, pad_right) for last dim
        else:
            pad = (pad_total, 0)
        return F.pad(waveform, pad, mode="constant", value=0.0)

target = PadOrTrimTo(16000)

dataset = SPEECHCOMMANDS(
    root="./data",
    subset="training",
    download=True,
    transform=target,
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

TypeError: SPEECHCOMMANDS.__init__() got an unexpected keyword argument 'transform'

In [23]:
from collections import Counter

counts = Counter()
longer_samples = []
shorter_samples = []

for i, (waveform, _, _, _, _) in enumerate(dataset):
    length = waveform.shape[1]
    if length != 16000:
        if length > 16000:
            counts["longer"] += 1
            longer_samples.append(i)
        else:
            counts["shorter"] += 1
            shorter_samples.append(i)

print(f"Total different length samples: {counts['longer'] + counts['shorter']}")
print(f"Longer: {counts['longer']}, Shorter: {counts['shorter']}")
print(f"Indices of longer samples: {longer_samples}")
print(f"Indices of shorter samples: {shorter_samples}")


Total different length samples: 8479
Longer: 0, Shorter: 8479
Indices of longer samples: []
Indices of shorter samples: [40, 56, 66, 87, 88, 90, 91, 101, 102, 113, 118, 119, 120, 121, 135, 176, 177, 178, 184, 200, 201, 348, 355, 356, 397, 401, 402, 403, 404, 464, 470, 471, 491, 515, 516, 522, 523, 539, 540, 585, 586, 620, 630, 652, 663, 678, 695, 696, 721, 722, 736, 770, 795, 796, 833, 843, 850, 851, 880, 881, 893, 894, 895, 896, 948, 1010, 1011, 1012, 1023, 1050, 1055, 1056, 1057, 1105, 1106, 1107, 1143, 1148, 1162, 1193, 1198, 1213, 1311, 1312, 1340, 1345, 1347, 1351, 1357, 1360, 1364, 1366, 1370, 1378, 1381, 1398, 1405, 1416, 1426, 1435, 1438, 1452, 1454, 1455, 1465, 1468, 1469, 1477, 1490, 1499, 1503, 1504, 1516, 1517, 1518, 1519, 1533, 1554, 1558, 1559, 1574, 1575, 1577, 1590, 1592, 1593, 1594, 1595, 1598, 1619, 1620, 1622, 1623, 1624, 1629, 1630, 1641, 1657, 1660, 1662, 1664, 1676, 1677, 1696, 1700, 1748, 1749, 1752, 1759, 1761, 1764, 1765, 1766, 1777, 1778, 1779, 1801, 1802, 180

In [ ]:
import torch
# zero pad or truncate
def pad_trunc(waveform, max_len=16000):
    length = waveform.shape[1]
    if length > max_len:
        waveform = waveform[:, :max_len]
    elif length < max_len:
        pad_amount = max_len - length
        waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
    return waveform

# add a transform to the dataset